# 🎮 Projet 03 : "Grand Consolidation Data Energy & Prétraitement"

Bienvenue dans le troisième projet ! Tu es promu **Ingénieur Principal Data & Audit Énergétique**.

L'objectif de ce projet est de **consolider 100 % des notions fondamentales** en résolvant un cas industriel réel : nettoyer des données brutes de capteurs altérées, appliquer un prétraitement robuste avec **NumPy** et **Pandas**, et construire un dashboard de visualisation complet avec **Matplotlib**.

---

## 🛠️ Le Pipeline de Prétraitement de Données (Data Preprocessing)

En ingénierie de données, 80 % du travail consiste à préparer la donnée brute avant toute analyse :

```text
[ Données Brutes Altérées ] 
          │
          ├──> 1. Détection & Suppression des Doublons
          ├──> 2. Correction des Types (Dates & Chiffres)
          ├──> 3. Imputation des Trous de Signal (NaN)
          ├──> 4. Calcul de Variables Physiques Vectorisées
          └──> 5. Agrégation & Visualisation Multi-Panneaux

## 🟢 Niveau 1 : NumPy Avancé (Matrices, Axes & Masquage Booléen)

### 📚 Cours théorique pas-à-pas :

#### 1. Le Masquage Booléen avec NumPy
Au lieu de parcourir un tableau 2D avec deux boucles `for` pour trouver des conditions, NumPy permet de créer un **masque booléen** instantané :

```python
import numpy as np

matrice = np.array([[220, 240], [210, 250]])
masque = matrice > 235  # Renvoie un array de True/False de même forme !
# Resultat : [[False, True], [False, True]]

#### 2. Calculs statistiques par Axe (axis=0 vs axis=1)
**`axis=0`** (Vertical / Colonnes) : Résultat par capteur sur tout l'historique.

**`axis=1`** (Horizontal / Lignes) : Résultat global par instant temporel.

```axis=0 (Moyenne par Capteur)
                   ↓      ↓      ↓
Ligne 0 (t=0h)  [ 220.0, 220.0, 230.0 ]  ──> axis=1 (Moyenne à t=0h)
Ligne 1 (t=1h)  [ 230.0, 225.0, 240.0 ]  ──> axis=1 (Moyenne à t=1h)
Ligne 2 (t=2h)  [ 210.0, 230.0, 250.0 ]  ──> axis=1 (Moyenne à t=2h)

In [ ]:
import numpy as np

# ==========================================
# QUÊTE 1 : Analyse Vectorielle de Surtensions
# ==========================================
# Matrice de tension (3 instants x 3 capteurs) en Volts :
tensions_reseau = np.array([
    [220.0, 220.0, 230.0],  # t = 0h
    [230.0, 225.0, 240.0],  # t = 1h
    [210.0, 230.0, 250.0]   # t = 2h
])

seuil_surtension = 235.0

# Consignes :
# 1. Crée un masque booléen `masque_surtension` identifiant les cases >= seuil_surtension.
# 2. Compte le nombre total de surtensions dans `nb_surtensions` avec `.sum()`.
# 3. Calcule la tension moyenne de chaque capteur sur la journée dans `moyenne_par_capteur` (axis=0).

# ### START CODE HERE ###

# ### END CODE HERE ###

# --- VALIDATION AUTOMATIQUE ---
assert nb_surtensions == 2, f"Attendu : 2 surtensions, obtenu : {nb_surtensions}"
assert np.array_equal(moyenne_par_capteur, np.array([220.0, 225.0, 240.0])), f"Erreur moyenne : {moyenne_par_capteur}"
print("🏆 Quête 1 réussie ! Masquage et agrégation NumPy maîtrisés.")

# --- AFFICHAGE VISUEL EN TEMPS RÉEL ---
from utils import tracer_masquage_numpy
tracer_masquage_numpy(tensions_reseau, masque_surtension)

## 🟡 Niveau 2 : Prétraitement Pandas (Doublons, Dates & Imputation `ffill`)

### 📚 Cours théorique pas-à-pas :

#### 1. Supprimer les Doublons (`.drop_duplicates()`)
Les réseaux IoT renvoient parfois plusieurs fois le même paquet suite à des retards de connexion.
```python
df = df.drop_duplicates()  # Supprime les lignes 100 % identiques

#### 2. Conversion Temporelle (pd.to_datetime())
Pour trier ou filtrer par date/heure, la colonne doit impérativement être convertie du type object (texte) vers le type **`datetime64`** :

```python
df['horodateur'] = pd.to_datetime(df['horodateur'])

#### 3. Réparer les NaN par Imputation Temporelle (`.ffill()`)
Pour combler les trous sans inventer de données incohérentes, on propage la dernière valeur connue :

```python
df['tension_v'] = df['tension_v'].ffill()

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# QUÊTE 2 : Nettoyage d'un Dataset Perturbé
# ==========================================
logs_corrompus = {
    'horodateur': ['2026-08-25 10:00', '2026-08-25 10:00', '2026-08-25 11:00', '2026-08-25 12:00', '2026-08-25 13:00'],
    'tension_v': [230.0, 230.0, np.nan, 225.0, 235.0],
    'courant_a': [10.0, 10.0, 12.0, np.nan, 15.0],
    'site': ['SITE_A', 'SITE_A', 'SITE_A', 'SITE_A', 'SITE_A']
}

df_brut = pd.DataFrame(logs_corrompus)

# Consignes :
# 1. Copie le dataframe dans `df_propre`.
# 2. Supprime les lignes doublons avec `.drop_duplicates()`.
# 3. Convertis la colonne 'horodateur' au format `pd.to_datetime()`.
# 4. Impute les NaN des colonnes 'tension_v' et 'courant_a' avec `.ffill()`.
# 5. Crée la colonne `puissance_kw` = (tension_v * courant_a) / 1000.0.

# ### START CODE HERE ###

# ### END CODE HERE ###

# --- VALIDATION AUTOMATIQUE ---
assert len(df_propre) == 4, f"Nombre de lignes incorrect après dédoublonnage : {len(df_propre)}"
assert df_propre['tension_v'].isna().sum() == 0, "Il reste des NaN dans tension_v"
assert round(df_propre['puissance_kw'].sum(), 3) == 11.285, f"Erreur calcul puissance : {df_propre['puissance_kw'].sum()}"
print("🏆 Quête 2 réussie ! Pipeline de nettoyage validé.")

# --- AFFICHAGE VISUEL EN TEMPS RÉEL ---
from utils import tracer_nettoyage_complet
tracer_nettoyage_complet(df_brut, df_propre)

## 🟠 Niveau 3 : Groupement Avancé & Multi-Agrégation (`.groupby().agg()`)

### 📚 Cours théorique pas-à-pas :

#### 1. L'Agrégation Multiple (`.agg()`)
Lorsque l'on regroupe des données par catégorie (ex: par site), on souhaite souvent appliquer **des calculs différents sur chaque colonne** (ex: la somme pour la puissance, la moyenne pour le rendement).

```python
# Exemple de syntaxe avec renommage direct des colonnes :
df_kpi = df.groupby('site').agg(
    puissance_totale_kw=('puissance_kw', 'sum'),
    rendement_moyen_pct=('rendement_pct', 'mean')
).reset_index()

In [ ]:
# ==========================================
# QUÊTE 3 : Consolidation Multi-Site
# ==========================================
donnees_multi_sites = {
    'site': ['CENTRALE_NORTH', 'CENTRALE_NORTH', 'CENTRALE_SOUTH', 'CENTRALE_SOUTH'],
    'puissance_kw': [120.0, 150.0, 200.0, 250.0],
    'rendement_pct': [92.0, 94.0, 88.0, 90.0]
}

df_sites = pd.DataFrame(donnees_multi_sites)

# Consignes :
# 1. Regroupe par 'site' et calcule :
#    - `puissance_totale_kw` : la SOMME de 'puissance_kw'
#    - `rendement_moyen_pct` : la MOYENNE de 'rendement_pct'
# 2. Stocke le résultat dans `df_bilan` avec `.reset_index()`.

# ### START CODE HERE ###

# ### END CODE HERE ###

# --- VALIDATION AUTOMATIQUE ---
assert len(df_bilan) == 2, "Il doit y avoir 2 sites consolidés"
val_north = df_bilan.loc[df_bilan['site'] == 'CENTRALE_NORTH', 'puissance_totale_kw'].values[0]
assert val_north == 270.0, f"Erreur puissance North : attendu 270.0, obtenu {val_north}"
print("🏆 Quête 3 réussie ! Consolidation multi-agrégation validée.")

# --- AFFICHAGE VISUEL EN TEMPS RÉEL ---
from utils import tracer_kpi_groupby
tracer_kpi_groupby(df_bilan)

## 🔴 Niveau 4 : Masterclass Matplotlib (Dashboards Multi-Panneaux)

### 📚 Cours théorique pas-à-pas :

#### 1. Structure d'une Figure Multi-Graphiques (`plt.add_gridspec` ou `plt.subplots`)
Pour présenter un rapport professionnel, on regroupe plusieurs graphiques dans une même figure :
* **`fig.add_subplot(gs[0, :])`** : Graphique occupant toute la première ligne.
* **`fig.add_subplot(gs[1, 0])`** : Graphique situé en bas à gauche.
* **`fig.add_subplot(gs[1, 1])`** : Graphique situé en bas à droite.

#### 2. Différents types de tracés :
* **`ax.plot()`** : Courbe temporelle.
* **`ax.hist()`** : Distribution statistique des mesures.
* **`ax.scatter()`** : Analyse de corrélation entre 2 variables physiques.

In [ ]:
# ==========================================
# QUÊTE 4 : Génération du Dashboard Multi-Panneaux
# ==========================================
# Utilisons df_propre de la Quête 2 pour générer le rapport visuel final.

# Consigne :
# Appelle la fonction `tracer_dashboard_matplotlib_complet(df_propre)` issue de utils.py.

# ### START CODE HERE ###
from utils import tracer_dashboard_matplotlib_complet

tracer_dashboard_matplotlib_complet(df_propre)
# ### END CODE HERE ###

print("🏆 Quête 4 réussie ! Dashboard multi-panneaux affiché.")

## 🟣 Niveau 5 : Grand Défi Synthèse (Le Pipeline Complet)

### 📚 Cours théorique pas-à-pas :

L'objectif final est de concevoir une **fonction Python modulaire et autonome** `traiter_donnees_microgrid(df_raw)` qui prend un dataset brut en entrée et exécute automatiquement toute la chaîne de traitement :
1. Suppression des doublons.
2. Imputation des valeurs manquantes (`.ffill()`).
3. Calcul des variables dérivées (`puissance_kw`).
4. Agrégation des résultats par site.

In [ ]:
# ==========================================
# QUÊTE 5 : La Fonction Pipeline Universelle
# ==========================================
# Dataset brut transmis par l'opérateur :
raw_data = pd.DataFrame({
    'horodateur': ['2026-08-25 08:00', '2026-08-25 08:00', '2026-08-25 09:00', '2026-08-25 09:00'],
    'site': ['PARC_SOLAIRE_1', 'PARC_SOLAIRE_1', 'PARC_SOLAIRE_2', 'PARC_SOLAIRE_2'],
    'tension_v': [220.0, 220.0, np.nan, 230.0],
    'courant_a': [10.0, 10.0, 15.0, 15.0]
})

# Consigne :
# Rédige la fonction `traiter_donnees_microgrid(df)` qui :
# 1. Supprime les doublons.
# 2. Applique `.ffill()` sur 'tension_v' et 'courant_a'.
# 3. Ajoute la colonne `puissance_kw` = (tension_v * courant_a) / 1000.0.
# 4. Renvoie le DataFrame nettoyé ET un DataFrame agrégé par 'site' donnant le total de `puissance_kw`.

# ### START CODE HERE ###

# ### END CODE HERE ###

# --- VALIDATION AUTOMATIQUE ---
df_clean, df_summary = traiter_donnees_microgrid(raw_data)

assert len(df_clean) == 3, f"Taille df_clean attendue : 3, obtenue : {len(df_clean)}"
assert df_clean['tension_v'].isna().sum() == 0, "Il reste des valeurs manquantes"
assert len(df_summary) == 2, "Il doit y avoir 2 sites résumé"
print("🏆 Quête 5 réussie ! Pipeline de traitement autonome validé.")

# --- AFFICHAGE VISUEL ET RAPPORT FINAL ---
from utils import afficher_bilan_grand_projet
afficher_bilan_grand_projet(df_clean, df_summary)

## 👑 Bilan Général du Projet 03

Félicitations ! Tu as consolidé avec succès les compétences fondamentales de l'ingénieur de données IoT :
1. **NumPy** : Masquage booléen et calculs statistiques par axes (`axis=0/1`).
2. **Pandas** : Pipeline de nettoyage complet (dédoublonnage, formats de dates, imputation `.ffill()`).
3. **Data Aggregation** : Bilan multi-indicateurs avec `.groupby().agg()`.
4. **Matplotlib** : Conception d'un dashboard de diagnostic multi-panneaux.
5. **Modularité** : Encapsulation du pipeline dans une fonction réutilisable.